In [0]:
from pyspark.sql.functions import col, current_timestamp
import re

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog Name")
dbutils.widgets.text("user_login", "w_k_palczewski", "User Login")
dbutils.widgets.text("table_name", "ingested_data", "Target Table Name")

In [0]:
catalog = dbutils.widgets.get("catalog")
user_login = dbutils.widgets.get("user_login")
table_name = dbutils.widgets.get("table_name")

In [0]:
base_volume_path = f"/Volumes/{catalog}/{user_login}/raw_data"

source_path = f"{base_volume_path}/data" 

checkpoint_path = f"{base_volume_path}/checkpoints/{table_name}"
schema_location = f"{base_volume_path}/schemas/{table_name}"

target_table = f"{catalog}.{user_login}_bronze.{table_name}"

In [0]:
df_raw = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaLocation", schema_location)
  .load(source_path)
)

In [0]:
clean_columns = [re.sub(r'[ ,;{}()\n\t=]+', '_', c).strip('_') for c in df_raw.columns]
df_cleaned = df_raw.toDF(*clean_columns)

df_bronze = (df_cleaned
  .withColumn("source_filename", col("_metadata.file_path"))
  .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
query = (df_bronze.writeStream
  .trigger(availableNow=True)
  .option("checkpointLocation", checkpoint_path)
  .toTable(target_table)
)

query.awaitTermination()